# Control Actions Sheet Enrichment

This notebook enriches the existing `control_actions` sheet in `DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx` without removing any original rows.

It adds:
- raw signed `Step`
- same-minute merged action details repeated across the member rows
- deviation start from `SSD_1071_650episodes_1April2026.csv`
- minutes from deviation to merged action
- merged action context columns
- Excel row styling for merged-group boundaries

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Border, PatternFill, Side

BASE_DIR = Path("/home/h604827/ControlActions")
WORKBOOK_PATH = BASE_DIR / "RESULTS/03LIC_1071_episodes_12JUN2026_0921/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx"
DEDUP_EVENTS_PATH = BASE_DIR / "DATA/combined_events/trip_filtered/03LIC_1071_PVLO_PVHI_combined_events.parquet"
SSD_PATH = BASE_DIR / "DATA/SSD_1071_ControlActions_24April2026.csv"
PV_PATH = BASE_DIR / "DATA/PV-OP_data/03LIC_1071_JAN_2026.parquet"
OPERATING_LIMITS_PATH = BASE_DIR / "DATA/operating_limits.csv"

TARGET_SSD_TAG = "03LIC_1071.PV"
ALARM_THRESHOLD = 28.75


def parse_mixed_value(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        return float(value)
    text = str(value).strip()
    if text == "":
        return np.nan
    try:
        number = float(text)
    except ValueError:
        return text
    if np.isfinite(number) and float(number).is_integer():
        return int(number)
    return number


def classify_direction(step_value):
    if pd.isna(step_value):
        return pd.NA
    if step_value > 0:
        return "up"
    if step_value < 0:
        return "down"
    return "none"


def classify_timing(timestamp, start, end):
    if pd.isna(timestamp) or pd.isna(start) or pd.isna(end):
        return pd.NA
    if timestamp < start:
        return "before"
    if timestamp > end:
        return "after"
    return "during"


def asof_snapshot(frame, timestamps):
    timestamp_series = pd.to_datetime(pd.Series(timestamps)).reset_index(drop=True)
    indexer = frame.index.get_indexer(timestamp_series, method="ffill")
    missing = indexer == -1
    if missing.any():
        indexer[missing] = frame.index.get_indexer(timestamp_series[missing], method="bfill")
    snapshot = frame.iloc[indexer].copy()
    snapshot.index = timestamp_series.index
    return snapshot


print(f"Workbook path: {WORKBOOK_PATH}")

Workbook path: /home/h604827/ControlActions/DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx


In [2]:
alarm_clusters_df = pd.read_excel(WORKBOOK_PATH, sheet_name="alarm_clusters")
control_actions_df = pd.read_excel(WORKBOOK_PATH, sheet_name="control_actions")

for col in ["VT_Start", "cluster_start", "cluster_end"]:
    control_actions_df[col] = pd.to_datetime(control_actions_df[col])

control_actions_df["original_row_number"] = np.arange(2, len(control_actions_df) + 2)

events_lookup_df = pd.read_csv(
    DEDUP_EVENTS_PATH,
    usecols=["VT_Start", "Source", "Description", "PrevValue", "Value"],
    low_memory=False,
)
events_lookup_df["VT_Start_src"] = pd.to_datetime(events_lookup_df["VT_Start"])
events_lookup_df["VT_ms"] = events_lookup_df["VT_Start_src"].dt.round("ms")
events_lookup_df = (
    events_lookup_df
    .drop_duplicates(subset=["Source", "Description", "VT_ms"])
    .rename(columns={"PrevValue": "PrevValue_src", "Value": "Value_src"})
    .drop(columns=["VT_Start"])
)

control_actions_df["VT_ms"] = control_actions_df["VT_Start"].dt.round("ms")
control_actions_df = control_actions_df.merge(
    events_lookup_df,
    on=["Source", "Description", "VT_ms"],
    how="left",
)

for col in ["PrevValue", "Value"]:
    restored_values = control_actions_df[f"{col}_src"].combine_first(control_actions_df[col])
    control_actions_df[col] = restored_values.map(parse_mixed_value).astype("object")

control_actions_df = control_actions_df.drop(columns=["VT_ms", "VT_Start_src", "PrevValue_src", "Value_src"])

control_actions_df["PrevValue_num"] = pd.to_numeric(control_actions_df["PrevValue"], errors="coerce")
control_actions_df["Value_num"] = pd.to_numeric(control_actions_df["Value"], errors="coerce")
control_actions_df["Step"] = control_actions_df["Value_num"] - control_actions_df["PrevValue_num"]
control_actions_df["minute_bucket"] = control_actions_df["VT_Start"].dt.floor("min")

eligible_mask = (
    control_actions_df["Description"].isin(["OP", "SP"])
    & control_actions_df["PrevValue_num"].notna()
    & control_actions_df["Value_num"].notna()
)

merged_base_columns = [
    "merged_group_id",
    "merged_group_role",
    "merged_action_timestamp",
    "merged_action_timing",
    "merged_prev_value",
    "merged_value",
    "merged_step",
    "merged_action_direction",
    "merged_num_actions",
    "deviation_start",
    "merged_minutes_from_deviation",
]

for col in merged_base_columns:
    control_actions_df[col] = pd.NA

eligible_actions_df = control_actions_df.loc[eligible_mask].drop(columns=merged_base_columns, errors="ignore").copy()
eligible_actions_df["_source_index"] = eligible_actions_df.index

group_keys = ["cluster_id", "Source", "Description", "minute_bucket"]
sort_columns = group_keys + ["VT_Start", "_source_index"]
eligible_actions_df = eligible_actions_df.sort_values(sort_columns)

eligible_actions_df["merged_group_position"] = eligible_actions_df.groupby(group_keys, sort=False).cumcount()
eligible_actions_df["merged_group_size"] = eligible_actions_df.groupby(group_keys, sort=False)["VT_Start"].transform("size")

group_meta_df = eligible_actions_df.groupby(group_keys, sort=False).agg(
    cluster_start=("cluster_start", "first"),
    cluster_end=("cluster_end", "first"),
    merged_action_timestamp=("VT_Start", "first"),
    merged_prev_value=("PrevValue_num", "first"),
    merged_value=("Value_num", "last"),
    merged_num_actions=("VT_Start", "size"),
).reset_index()

group_meta_df["merged_step"] = group_meta_df["merged_value"] - group_meta_df["merged_prev_value"]
group_meta_df["merged_action_direction"] = group_meta_df["merged_step"].map(classify_direction)
group_meta_df["merged_action_timing"] = group_meta_df.apply(
    lambda row: classify_timing(row["merged_action_timestamp"], row["cluster_start"], row["cluster_end"]),
    axis=1,
)
group_meta_df["merged_group_id"] = [f"MG_{idx:05d}" for idx in range(1, len(group_meta_df) + 1)]

eligible_actions_df = eligible_actions_df.merge(
    group_meta_df[
        group_keys
        + [
            "merged_group_id",
            "merged_action_timestamp",
            "merged_action_timing",
            "merged_prev_value",
            "merged_value",
            "merged_step",
            "merged_action_direction",
            "merged_num_actions",
        ]
    ],
    on=group_keys,
    how="left",
)

eligible_actions_df["merged_group_role"] = np.select(
    [
        eligible_actions_df["merged_group_size"] == 1,
        eligible_actions_df["merged_group_position"] == 0,
        eligible_actions_df["merged_group_position"] == eligible_actions_df["merged_group_size"] - 1,
    ],
    ["SINGLE", "START", "END"],
    default="BODY",
)

raw_step_count = control_actions_df["Step"].notna().sum()
merged_group_count = len(group_meta_df)
multi_action_group_count = int((group_meta_df["merged_num_actions"] > 1).sum())

print(f"Original control action rows: {len(control_actions_df):,}")
print(f"Eligible numeric OP/SP rows: {eligible_mask.sum():,}")
print(f"Rows with raw Step populated: {raw_step_count:,}")
print(f"Merged action groups: {merged_group_count:,}")
print(f"Merged groups with more than one row: {multi_action_group_count:,}")

group_meta_df.head()

Original control action rows: 16,094
Eligible numeric OP/SP rows: 14,456
Rows with raw Step populated: 14,457
Merged action groups: 7,246
Merged groups with more than one row: 2,366


,cluster_id,Source,Description,minute_bucket,cluster_start,cluster_end,merged_action_timestamp,merged_prev_value,merged_value,merged_num_actions,merged_step,merged_action_direction,merged_action_timing,merged_group_id
0,1,03FIC_1085,OP,2022-01-05 08:58:00,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,2022-01-05 08:58:44.656,24.3784,18.3784,3,-6.0,down,during,MG_00001
1,1,03FIC_1085,OP,2022-01-05 09:29:00,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,2022-01-05 09:29:54.807,18.3784,28.3784,5,10.0,up,during,MG_00002
2,1,03FIC_1085,OP,2022-01-05 09:30:00,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,2022-01-05 09:30:00.218,28.3784,42.3784,7,14.0,up,during,MG_00003
3,1,03FIC_1085,OP,2022-01-05 09:31:00,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,2022-01-05 09:31:01.469,42.3784,44.3784,1,2.0,up,during,MG_00004
4,1,03FIC_1085,OP,2022-01-05 09:34:00,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,2022-01-05 09:34:08.056,44.3784,38.3784,3,-6.0,down,after,MG_00005


In [6]:
ssd_df = pd.read_csv(SSD_PATH, low_memory=False)
for col in ["AlarmStart_rounded", "AlarmEnd_rounded", "First_Transition_Start_Time"]:
    ssd_df[col] = pd.to_datetime(ssd_df[col])

ssd_df["ssd_alarm_start_floor"] = ssd_df["AlarmStart_rounded"].dt.floor("min")
ssd_df["ssd_alarm_end_floor"] = ssd_df["AlarmEnd_rounded"].dt.floor("min")

ssd_alarm_min_df = (
    ssd_df.dropna(subset=["First_Transition_Start_Time"])
    .groupby(["ssd_alarm_start_floor", "ssd_alarm_end_floor"], as_index=False)["First_Transition_Start_Time"]
    .min()
    .rename(columns={"First_Transition_Start_Time": "deviation_start"})
)

cluster_deviation_df = (
    group_meta_df[["cluster_id", "cluster_start", "cluster_end"]]
    .drop_duplicates()
    .copy()
)
cluster_deviation_df["cluster_start_floor"] = cluster_deviation_df["cluster_start"].dt.round("s").dt.floor("min")
cluster_deviation_df["cluster_end_floor"] = cluster_deviation_df["cluster_end"].dt.round("s").dt.floor("min")
cluster_deviation_df = cluster_deviation_df.merge(
    ssd_alarm_min_df,
    left_on=["cluster_start_floor", "cluster_end_floor"],
    right_on=["ssd_alarm_start_floor", "ssd_alarm_end_floor"],
    how="left",
)
cluster_deviation_df["has_ssd_match"] = cluster_deviation_df["deviation_start"].notna()

missing_alarm_clusters_df = (
    cluster_deviation_df.loc[
        ~cluster_deviation_df["has_ssd_match"],
        ["cluster_id", "cluster_start", "cluster_end", "cluster_start_floor", "cluster_end_floor"],
    ]
    .sort_values(["cluster_start", "cluster_id"])
    .reset_index(drop=True)
)

print(f"Distinct alarms matched to SSD: {cluster_deviation_df['has_ssd_match'].sum():,} / {len(cluster_deviation_df):,}")
print(f"Distinct alarms missing SSD match: {len(missing_alarm_clusters_df):,}")
if not missing_alarm_clusters_df.empty:
    print("Alarms with no SSD match will keep missing deviation_start values.")
    display(missing_alarm_clusters_df)

group_meta_df = group_meta_df.drop(
    columns=["deviation_start", "has_ssd_match", "merged_minutes_from_deviation"],
    errors="ignore",
)
group_meta_df = group_meta_df.merge(
    cluster_deviation_df[
        [
            "cluster_id",
            "cluster_start",
            "cluster_end",
            "deviation_start",
            "has_ssd_match",
        ]
    ],
    on=["cluster_id", "cluster_start", "cluster_end"],
    how="left",
)
group_meta_df["merged_minutes_from_deviation"] = np.where(
    group_meta_df["deviation_start"].notna(),
    (group_meta_df["merged_action_timestamp"] - group_meta_df["deviation_start"]).dt.total_seconds() / 60.0,
    np.nan,
)

pv_op_data_df = pd.read_parquet(PV_PATH)
if "TimeStamp" in pv_op_data_df.columns:
    pv_op_data_df["TimeStamp"] = pd.to_datetime(pv_op_data_df["TimeStamp"])
    pv_op_data_df = pv_op_data_df.set_index("TimeStamp")
pv_op_data_df.index = pd.to_datetime(pv_op_data_df.index)
pv_op_data_df = pv_op_data_df.sort_index()

operating_limits_df = pd.read_csv(OPERATING_LIMITS_PATH)
operating_limits_df = operating_limits_df[operating_limits_df["TAG_NAME"].astype(str).str.endswith(".PV")].copy()
operating_limits_df["range"] = operating_limits_df["UPPER_LIMIT"] - operating_limits_df["LOWER_LIMIT"]
operating_limits_df = operating_limits_df[operating_limits_df["range"] > 0].drop_duplicates(subset=["TAG_NAME"])
operating_limits_df = operating_limits_df.set_index("TAG_NAME")

context_pv_tags = [col for col in pv_op_data_df.columns if col.endswith(".PV")]
valid_pv_tags = [tag for tag in context_pv_tags if tag in operating_limits_df.index]

lower_bounds = operating_limits_df.loc[valid_pv_tags, "LOWER_LIMIT"]
ranges = operating_limits_df.loc[valid_pv_tags, "range"]

action_snapshot = asof_snapshot(pv_op_data_df[valid_pv_tags], group_meta_df["merged_action_timestamp"])
deviation_snapshot = pd.DataFrame(np.nan, index=group_meta_df.index, columns=valid_pv_tags)
valid_deviation_mask = group_meta_df["deviation_start"].notna()
if valid_deviation_mask.any():
    deviation_snapshot.loc[valid_deviation_mask, valid_pv_tags] = asof_snapshot(
        pv_op_data_df[valid_pv_tags],
        group_meta_df.loc[valid_deviation_mask, "deviation_start"],
    ).values
three_min_snapshot = asof_snapshot(
    pv_op_data_df[valid_pv_tags],
    group_meta_df["merged_action_timestamp"] - pd.Timedelta(minutes=3),
)
five_min_snapshot = asof_snapshot(
    pv_op_data_df[valid_pv_tags],
    group_meta_df["merged_action_timestamp"] - pd.Timedelta(minutes=5),
)

norm_pos_df = (action_snapshot[valid_pv_tags] - lower_bounds.values) / ranges.values
episode_norm_roc_df = (action_snapshot[valid_pv_tags] - deviation_snapshot[valid_pv_tags]) / ranges.values
local_3m_delta_df = (action_snapshot[valid_pv_tags] - three_min_snapshot[valid_pv_tags]) / ranges.values
local_5m_delta_df = (action_snapshot[valid_pv_tags] - five_min_snapshot[valid_pv_tags]) / ranges.values


def rename_context_columns(frame, suffix):
    renamed = frame.copy()
    renamed.columns = [f"merged_ctx_{col.replace('.PV', '')}_{suffix}" for col in renamed.columns]
    return renamed


context_feature_df = pd.concat(
    [
        rename_context_columns(norm_pos_df, "norm_pos"),
        rename_context_columns(episode_norm_roc_df, "episode_norm_roc"),
        rename_context_columns(local_3m_delta_df, "local_3m_delta_norm"),
        rename_context_columns(local_5m_delta_df, "local_5m_delta_norm"),
    ],
    axis=1,
)

if TARGET_SSD_TAG in action_snapshot.columns and TARGET_SSD_TAG in operating_limits_df.index:
    target_upper = operating_limits_df.loc[TARGET_SSD_TAG, "UPPER_LIMIT"]
    context_feature_df["merged_ctx_03LIC_1071_pv_at_action"] = action_snapshot[TARGET_SSD_TAG].values
    context_feature_df["merged_ctx_alarm_proximity"] = (
        action_snapshot[TARGET_SSD_TAG].values - ALARM_THRESHOLD
    ) / (target_upper - ALARM_THRESHOLD)

median_cluster_duration_minutes = (
    group_meta_df["cluster_end"] - group_meta_df["cluster_start"]
).dt.total_seconds().median() / 60.0
context_feature_df["merged_ctx_time_progress_ratio"] = (
    group_meta_df["merged_minutes_from_deviation"] / median_cluster_duration_minutes
)

group_export_columns = [
    "merged_group_id",
    "merged_action_timestamp",
    "merged_action_timing",
    "merged_prev_value",
    "merged_value",
    "merged_step",
    "merged_action_direction",
    "merged_num_actions",
    "deviation_start",
    "merged_minutes_from_deviation",
]
group_export_df = pd.concat(
    [
        group_meta_df[group_export_columns].reset_index(drop=True),
        context_feature_df.reset_index(drop=True),
    ],
    axis=1,
)

eligible_export_df = eligible_actions_df[["_source_index", "merged_group_id", "merged_group_role"]].merge(
    group_export_df,
    on="merged_group_id",
    how="left",
)

assign_columns = [col for col in eligible_export_df.columns if col != "_source_index"]
control_actions_df.loc[eligible_export_df["_source_index"], assign_columns] = eligible_export_df[assign_columns].to_numpy()

final_drop_columns = ["PrevValue_num", "Value_num", "original_row_number"]
final_control_actions_df = control_actions_df.drop(columns=final_drop_columns)

ordered_columns = []
for col in [
    "cluster_id",
    "cluster_start",
    "cluster_end",
    "action_timing",
    "action_direction",
    "Source",
    "Description",
    "VT_Start",
    "PrevValue",
    "Value",
    "Step",
    "minute_bucket",
    "merged_group_id",
    "merged_group_role",
    "merged_action_timestamp",
    "merged_action_timing",
    "merged_prev_value",
    "merged_value",
    "merged_step",
    "merged_action_direction",
    "merged_num_actions",
    "deviation_start",
    "merged_minutes_from_deviation",
]:
    if col in final_control_actions_df.columns:
        ordered_columns.append(col)

context_columns = [col for col in final_control_actions_df.columns if col.startswith("merged_ctx_")]
remaining_columns = [
    col for col in final_control_actions_df.columns
    if col not in ordered_columns + context_columns
]
final_control_actions_df = final_control_actions_df[ordered_columns + context_columns + remaining_columns]

sample_group_ids = group_meta_df.loc[group_meta_df["merged_num_actions"] > 1, "merged_group_id"]
if not sample_group_ids.empty:
    sample_group_id = sample_group_ids.iloc[0]
    sample_columns = [
        "cluster_id",
        "Source",
        "Description",
        "VT_Start",
        "PrevValue",
        "Value",
        "Step",
        "merged_group_id",
        "merged_group_role",
        "merged_action_timestamp",
        "merged_prev_value",
        "merged_value",
        "merged_step",
        "merged_num_actions",
        "deviation_start",
        "merged_minutes_from_deviation",
    ]
    display(final_control_actions_df.loc[final_control_actions_df["merged_group_id"] == sample_group_id, sample_columns])

print(f"Context PV tags used: {len(valid_pv_tags)}")
print(f"Final control_actions row count: {len(final_control_actions_df):,}")
print(f"Final control_actions column count: {len(final_control_actions_df.columns):,}")

Distinct alarms matched to SSD: 404 / 419
Distinct alarms missing SSD match: 15
Alarms with no SSD match will keep missing deviation_start values.


,cluster_id,cluster_start,cluster_end,cluster_start_floor,cluster_end_floor
0,22,2022-02-09 18:09:51.306,2022-02-09 18:17:25.305,2022-02-09 18:09:00,2022-02-09 18:17:00
1,65,2022-04-19 02:44:02.103,2022-04-19 03:34:12.608,2022-04-19 02:44:00,2022-04-19 03:34:00
2,72,2022-06-03 00:10:39.553,2022-06-03 00:49:41.553,2022-06-03 00:10:00,2022-06-03 00:49:00
3,113,2022-11-24 20:06:08.303,2022-11-24 20:18:18.305,2022-11-24 20:06:00,2022-11-24 20:18:00
4,137,2023-02-10 03:40:16.254,2023-02-10 03:55:55.754,2023-02-10 03:40:00,2023-02-10 03:55:00
5,138,2023-03-04 09:42:42.554,2023-03-04 09:48:40.055,2023-03-04 09:42:00,2023-03-04 09:48:00
6,170,2023-04-30 02:16:13.504,2023-04-30 02:34:53.502,2023-04-30 02:16:00,2023-04-30 02:34:00
7,179,2023-06-05 10:12:24.502,2023-06-05 11:29:44.753,2023-06-05 10:12:00,2023-06-05 11:29:00
8,209,2023-10-25 01:45:14.104,2023-10-25 01:58:30.353,2023-10-25 01:45:00,2023-10-25 01:58:00
9,210,2023-10-25 02:46:07.104,2023-10-25 02:55:15.102,2023-10-25 02:46:00,2023-10-25 02:55:00


,cluster_id,Source,Description,VT_Start,PrevValue,Value,Step,merged_group_id,merged_group_role,merged_action_timestamp,merged_prev_value,merged_value,merged_step,merged_num_actions,deviation_start,merged_minutes_from_deviation
127,1,03FIC_1085,OP,2022-01-05 08:58:44.656,24.3784,22.3784,-2.0,MG_00001,START,2022-01-05 08:58:44.656000,24.3784,18.3784,-6.0,3,2022-01-05 07:29:00,89.744267
128,1,03FIC_1085,OP,2022-01-05 08:58:46.766,22.3784,20.3784,-2.0,MG_00001,BODY,2022-01-05 08:58:44.656000,24.3784,18.3784,-6.0,3,2022-01-05 07:29:00,89.744267
129,1,03FIC_1085,OP,2022-01-05 08:58:58.806,20.3784,18.3784,-2.0,MG_00001,END,2022-01-05 08:58:44.656000,24.3784,18.3784,-6.0,3,2022-01-05 07:29:00,89.744267


Context PV tags used: 26
Final control_actions row count: 16,094
Final control_actions column count: 130


In [5]:
from datetime import datetime
import shutil

from openpyxl import load_workbook
from openpyxl.styles import Border, PatternFill, Side
from openpyxl.utils import get_column_letter

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

source_workbook = load_workbook(WORKBOOK_PATH, read_only=True)
original_sheet_names = list(source_workbook.sheetnames)
source_workbook.close()

timestamp_label = datetime.now().strftime("%Y%m%d_%H%M%S")
output_workbook_path = WORKBOOK_PATH.with_name(
    f"{WORKBOOK_PATH.stem}_enriched_{timestamp_label}{WORKBOOK_PATH.suffix}"
)

shutil.copy2(WORKBOOK_PATH, output_workbook_path)

with pd.ExcelWriter(
    output_workbook_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer:
    alarm_clusters_df.to_excel(writer, sheet_name="alarm_clusters", index=False)
    final_control_actions_df.to_excel(writer, sheet_name="control_actions", index=False)

    worksheet = writer.sheets["control_actions"]
    max_row = len(final_control_actions_df) + 1
    max_col = len(final_control_actions_df.columns)
    role_col_idx = final_control_actions_df.columns.get_loc("merged_group_role") + 1
    visual_end_col_idx = final_control_actions_df.columns.get_loc("merged_minutes_from_deviation") + 1

    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = f"A1:{get_column_letter(max_col)}{max_row}"

    start_fill = PatternFill(fill_type="solid", fgColor="FFF2CC")
    end_fill = PatternFill(fill_type="solid", fgColor="DDEBF7")
    single_fill = PatternFill(fill_type="solid", fgColor="E2F0D9")
    accent_side = Side(style="thick", color="C65911")

    rows_to_style = [
        (row_idx, role)
        for row_idx, role in enumerate(final_control_actions_df["merged_group_role"], start=2)
        if role in {"START", "END", "SINGLE"}
    ]

    print(
        f"Styling {len(rows_to_style):,} boundary rows across the first {visual_end_col_idx} columns..."
    )

    iterator = tqdm(rows_to_style, desc="Styling merged groups") if tqdm is not None else rows_to_style

    for row_idx, role in iterator:
        if role == "START":
            fill = start_fill
            top_side = accent_side
            bottom_side = None
        elif role == "END":
            fill = end_fill
            top_side = None
            bottom_side = accent_side
        else:
            fill = single_fill
            top_side = accent_side
            bottom_side = accent_side

        for col_idx in range(1, visual_end_col_idx + 1):
            cell = worksheet.cell(row=row_idx, column=col_idx)
            cell.border = Border(
                left=cell.border.left,
                right=cell.border.right,
                top=top_side or cell.border.top,
                bottom=bottom_side or cell.border.bottom,
            )

        worksheet.cell(row=row_idx, column=role_col_idx).fill = fill

validation_workbook = load_workbook(output_workbook_path, read_only=True)
output_sheet_names = list(validation_workbook.sheetnames)
validation_workbook.close()

validation_df = pd.read_excel(output_workbook_path, sheet_name="control_actions")
print(f"Created workbook copy: {output_workbook_path.name}")
print(f"Original workbook sheets: {', '.join(original_sheet_names)}")
print(f"Output workbook sheets: {', '.join(output_sheet_names)}")
print(f"Validated row count after write: {len(validation_df):,}")
print(validation_df["merged_group_role"].value_counts(dropna=False).head(10).to_string())

Styling 9,612 boundary rows across the first 23 columns...


Styling merged groups: 100%|██████████| 9612/9612 [00:07<00:00, 1276.64it/s]


Created workbook copy: 1071_pvlo_alarms_clustered_with_control_actions_enriched_20260427_085843.xlsx
Original workbook sheets: alarm_clusters, control_actions, overlapping_alarms
Output workbook sheets: alarm_clusters, control_actions, overlapping_alarms
Validated row count after write: 16,094
merged_group_role
SINGLE    4880
BODY      4844
START     2366
END       2366
NaN       1638


In [24]:
ignored_ssd_alarm_clusters_df = (
    missing_alarm_clusters_df[
        ["cluster_id", "cluster_start", "cluster_end", "cluster_start_floor", "cluster_end_floor"]
    ]
    .copy()
    .sort_values(["cluster_start", "cluster_id"])
    .reset_index(drop=True)
)

ignored_ssd_alarm_cluster_ids = ignored_ssd_alarm_clusters_df["cluster_id"].tolist()

ignored_ssd_alarm_group_df = (
    group_meta_df.loc[
        group_meta_df["cluster_id"].isin(ignored_ssd_alarm_cluster_ids),
        [
            "cluster_id",
            "cluster_start",
            "Source",
            "Description",
            "merged_group_id",
            "merged_action_timestamp",
            "merged_num_actions",
        ],
    ]
    .copy()
    .sort_values(["cluster_start", "Source", "merged_action_timestamp"])
    .reset_index(drop=True)
)

ignored_ssd_alarm_raw_row_count = (
    int(ignored_ssd_alarm_group_df["merged_num_actions"].sum())
    if not ignored_ssd_alarm_group_df.empty
    else 0
)

ignored_ssd_alarm_summary_df = pd.DataFrame(
    {
        "metric": [
            "Distinct alarms with no SSD match",
            "Merged groups inside ignored alarms",
            "Raw action rows inside ignored alarms",
        ],
        "value": [
            len(ignored_ssd_alarm_clusters_df),
            len(ignored_ssd_alarm_group_df),
            ignored_ssd_alarm_raw_row_count,
        ],
    }
)

print("SSD note for the control_actions context workbook")
print(
    "These alarms have no SSD match after the timestamp normalization fix. "
    "We will keep their workbook rows, leave SSD-backed deviation fields blank, and ignore these 15 alarms in downstream interpretation."
)
display(ignored_ssd_alarm_summary_df)

if not ignored_ssd_alarm_clusters_df.empty:
    print("\nIgnored alarms with no SSD match:")
    display(
        ignored_ssd_alarm_clusters_df[
            ["cluster_id", "cluster_start", "cluster_end", "cluster_start_floor", "cluster_end_floor"]
        ]
    )

SSD note for the control_actions context workbook
These alarms have no SSD match after the timestamp normalization fix. We will keep their workbook rows, leave SSD-backed deviation fields blank, and ignore these 15 alarms in downstream interpretation.


,metric,value
0,Distinct alarms with no SSD match,15
1,Merged groups inside ignored alarms,612
2,Raw action rows inside ignored alarms,1728



Ignored alarms with no SSD match:


,cluster_id,cluster_start,cluster_end,cluster_start_floor,cluster_end_floor
0,22,2022-02-09 18:09:51.306,2022-02-09 18:17:25.305,2022-02-09 18:09:00,2022-02-09 18:17:00
1,65,2022-04-19 02:44:02.103,2022-04-19 03:34:12.608,2022-04-19 02:44:00,2022-04-19 03:34:00
2,72,2022-06-03 00:10:39.553,2022-06-03 00:49:41.553,2022-06-03 00:10:00,2022-06-03 00:49:00
3,113,2022-11-24 20:06:08.303,2022-11-24 20:18:18.305,2022-11-24 20:06:00,2022-11-24 20:18:00
4,137,2023-02-10 03:40:16.254,2023-02-10 03:55:55.754,2023-02-10 03:40:00,2023-02-10 03:55:00
5,138,2023-03-04 09:42:42.554,2023-03-04 09:48:40.055,2023-03-04 09:42:00,2023-03-04 09:48:00
6,170,2023-04-30 02:16:13.504,2023-04-30 02:34:53.502,2023-04-30 02:16:00,2023-04-30 02:34:00
7,179,2023-06-05 10:12:24.502,2023-06-05 11:29:44.753,2023-06-05 10:12:00,2023-06-05 11:29:00
8,209,2023-10-25 01:45:14.104,2023-10-25 01:58:30.353,2023-10-25 01:45:00,2023-10-25 01:58:00
9,210,2023-10-25 02:46:07.104,2023-10-25 02:55:15.102,2023-10-25 02:46:00,2023-10-25 02:55:00


## Preliminary Analysis: 03LIC_1071 OP/SP Plant Context

This section looks only at unique merged actions for `03LIC_1071` and compares the plant context for `OP` versus `SP` actions.

Metrics used here:
- `merged_ctx_03LIC_1071_pv_at_action`: the actual `03LIC_1071.PV` value when the merged action starts
- `merged_ctx_alarm_proximity`: normalized distance from the low alarm threshold; `0` is exactly at the alarm threshold, positive values are above it, negative values are below it
- `merged_ctx_time_progress_ratio`: how late the action occurred relative to the median cluster duration; values below `1` are earlier, values above `1` are later
- `merged_ctx_03LIC_1071_norm_pos`: the PV position within the operating range at action time
- `merged_ctx_03LIC_1071_episode_norm_roc`: normalized change from deviation start to action time
- `merged_ctx_03LIC_1071_local_3m_delta_norm`: normalized PV change over the last 3 minutes before the action
- `merged_ctx_03LIC_1071_local_5m_delta_norm`: normalized PV change over the last 5 minutes before the action

In [9]:
analysis_metric_columns = [
    "merged_ctx_03LIC_1071_pv_at_action",
    "merged_ctx_alarm_proximity",
    "merged_ctx_time_progress_ratio",
    "merged_ctx_03LIC_1071_norm_pos",
    "merged_ctx_03LIC_1071_episode_norm_roc",
    "merged_ctx_03LIC_1071_local_3m_delta_norm",
    "merged_ctx_03LIC_1071_local_5m_delta_norm",
]

analysis_base_columns = [
    "cluster_id",
    "Source",
    "Description",
    "merged_group_id",
    "merged_action_timestamp",
    "merged_num_actions",
    "merged_step",
    "merged_action_direction",
    "deviation_start",
    "merged_minutes_from_deviation",
] + analysis_metric_columns

merged_1071_actions_df = (
    final_control_actions_df[
        (final_control_actions_df["Source"] == "03LIC_1071")
        & (final_control_actions_df["Description"].isin(["OP", "SP"]))
        & final_control_actions_df["merged_group_id"].notna()
    ][analysis_base_columns]
    .drop_duplicates(subset=["merged_group_id"])
    .sort_values(["Description", "merged_action_timestamp"])
    .reset_index(drop=True)
)

print(f"Unique merged 03LIC_1071 actions: {len(merged_1071_actions_df):,}")
print("Action counts by type:")
print(merged_1071_actions_df["Description"].value_counts().to_string())

display(merged_1071_actions_df.head(10))

Unique merged 03LIC_1071 actions: 553
Action counts by type:
Description
OP    355
SP    198


,cluster_id,Source,Description,merged_group_id,merged_action_timestamp,merged_num_actions,merged_step,merged_action_direction,deviation_start,merged_minutes_from_deviation,merged_ctx_03LIC_1071_pv_at_action,merged_ctx_alarm_proximity,merged_ctx_time_progress_ratio,merged_ctx_03LIC_1071_norm_pos,merged_ctx_03LIC_1071_episode_norm_roc,merged_ctx_03LIC_1071_local_3m_delta_norm,merged_ctx_03LIC_1071_local_5m_delta_norm
0,80,03LIC_1071,OP,MG_01258,2022-06-21 12:49:10.756000,15,-36.4097,down,2022-06-21 11:21:00,88.179267,65.337296,2.677474,2.212546,4.198716,3.850661,2.630919,5.235310
1,80,03LIC_1071,OP,MG_01259,2022-06-21 13:34:48.153000,1,-38.7079,down,2022-06-21 11:21:00,133.80255,4.296341,-1.789529,3.357301,-4.319255,-4.667310,0.670061,0.669802
2,80,03LIC_1071,OP,MG_01260,2022-06-21 13:37:35.513000,1,5.0,up,2022-06-21 11:21:00,136.591883,13.061336,-1.148103,3.427290,-3.096142,-3.444197,1.223113,1.893262
3,80,03LIC_1071,OP,MG_01261,2022-06-21 13:39:26.233000,1,5.0,up,2022-06-21 11:21:00,138.437217,7.724250,-1.538674,3.473592,-3.840907,-4.188962,-1.029080,0.478348
4,80,03LIC_1071,OP,MG_01262,2022-06-21 13:44:13.615000,2,15.0,up,2022-06-21 11:21:00,143.226917,1.851918,-1.968413,3.593772,-4.660362,-5.008418,-0.367616,-0.819456
5,80,03LIC_1071,OP,MG_01263,2022-06-21 14:10:08.353000,1,-25.0,down,2022-06-21 11:21:00,169.139217,29.515049,0.055987,4.243950,-0.800106,-1.148161,1.984320,3.001572
6,80,03LIC_1071,OP,MG_01264,2022-06-21 14:13:53.253000,1,-22.398,down,2022-06-21 11:21:00,172.88755,27.029156,-0.125932,4.338001,-1.147000,-1.495056,-0.346894,1.105100
7,80,03LIC_1071,OP,MG_01265,2022-06-21 15:34:17.453000,3,35.0,up,2022-06-21 11:21:00,253.290883,-0.508247,-2.141131,6.355438,-4.989712,-5.337767,0.000000,0.000000
8,80,03LIC_1071,OP,MG_01266,2022-06-21 15:44:47.703000,1,5.0,up,2022-06-21 11:21:00,263.79505,-0.508247,-2.141131,6.619003,-4.989712,-5.337767,0.000000,0.000000
9,80,03LIC_1071,OP,MG_01267,2022-06-21 15:46:36.455000,1,5.0,up,2022-06-21 11:21:00,265.607583,-0.508247,-2.141131,6.664482,-4.989712,-5.337767,0.000000,0.000000


In [10]:
metric_label_map = {
    "merged_ctx_03LIC_1071_pv_at_action": "1071 PV at action",
    "merged_ctx_alarm_proximity": "Alarm proximity",
    "merged_ctx_time_progress_ratio": "Time progress ratio",
    "merged_ctx_03LIC_1071_norm_pos": "1071 normalized position",
    "merged_ctx_03LIC_1071_episode_norm_roc": "1071 episode normalized change",
    "merged_ctx_03LIC_1071_local_3m_delta_norm": "1071 local 3 min normalized change",
    "merged_ctx_03LIC_1071_local_5m_delta_norm": "1071 local 5 min normalized change",
}

summary_frames = []
for metric in analysis_metric_columns:
    summary = (
        merged_1071_actions_df.groupby("Description")[metric]
        .agg(
            count="count",
            mean="mean",
            median="median",
            std="std",
            min="min",
            q25=lambda values: values.quantile(0.25),
            q75=lambda values: values.quantile(0.75),
            max="max",
        )
        .reset_index()
    )
    summary.insert(0, "metric", metric_label_map[metric])
    summary_frames.append(summary)

summary_stats_1071_df = pd.concat(summary_frames, ignore_index=True)
display(summary_stats_1071_df)

insight_rows = []
for action_type, subset in merged_1071_actions_df.groupby("Description"):
    insight_rows.append(
        {
            "Description": action_type,
            "merged_actions": len(subset),
            "median_merged_step": subset["merged_step"].median(),
            "pct_below_alarm_threshold": (subset["merged_ctx_03LIC_1071_pv_at_action"] < ALARM_THRESHOLD).mean() * 100,
            "pct_alarm_proximity_negative": (subset["merged_ctx_alarm_proximity"] < 0).mean() * 100,
            "pct_local_3m_rising": (subset["merged_ctx_03LIC_1071_local_3m_delta_norm"] > 0).mean() * 100,
            "pct_local_5m_rising": (subset["merged_ctx_03LIC_1071_local_5m_delta_norm"] > 0).mean() * 100,
            "median_minutes_from_deviation": subset["merged_minutes_from_deviation"].median(),
            "median_time_progress_ratio": subset["merged_ctx_time_progress_ratio"].median(),
        }
    )

insight_1071_df = pd.DataFrame(insight_rows)
display(insight_1071_df)

,metric,Description,count,mean,median,std,min,q25,q75,max
0,1071 PV at action,OP,353,34.275772,29.378994,32.558611,-1.327635,1.851918,58.373955,103.130840
1,1071 PV at action,SP,198,37.884692,33.492417,18.004029,-1.299326,27.621341,52.544258,75.076120
2,Alarm proximity,OP,353,0.404378,0.046030,2.382653,-2.201094,-1.968413,2.167894,5.443222
3,Alarm proximity,SP,198,0.668481,0.347052,1.317543,-2.199023,-0.082596,1.741274,3.390166
4,Time progress ratio,OP,355,3.530545,3.095418,1.682140,0.351679,2.446733,4.260551,8.708610
5,Time progress ratio,SP,194,3.783050,3.464109,1.732734,1.531632,2.560190,4.380713,12.323714
6,1071 normalized position,OP,353,-0.135770,-0.819091,4.543397,-5.104053,-4.660362,3.227016,9.472623
7,1071 normalized position,SP,198,0.367838,-0.245083,2.512376,-5.100103,-1.064363,2.413509,5.557722
8,1071 episode normalized change,OP,353,-1.225937,-1.148161,5.561662,-13.816228,-4.850712,2.520339,11.585866
9,1071 episode normalized change,SP,194,0.035477,-0.166048,3.478391,-14.391627,-1.574921,2.604083,8.900161


,Description,merged_actions,median_merged_step,pct_below_alarm_threshold,pct_alarm_proximity_negative,pct_local_3m_rising,pct_local_5m_rising,median_minutes_from_deviation,median_time_progress_ratio
0,OP,355,2.0,49.295775,49.295775,39.154930,40.563380,123.365417,3.095418
1,SP,198,1.0,34.848485,34.848485,45.959596,51.010101,138.059283,3.464109


In [11]:
def print_metric_interpretation(action_type, metric_column, friendly_name):
    subset = merged_1071_actions_df[merged_1071_actions_df["Description"] == action_type]
    if subset.empty:
        return
    series = subset[metric_column].dropna()
    if series.empty:
        print(f"{action_type} | {friendly_name}: no available values")
        return
    print(
        f"{action_type} | {friendly_name}: median={series.median():.4f}, "
        f"IQR=({series.quantile(0.25):.4f}, {series.quantile(0.75):.4f}), "
        f"range=({series.min():.4f}, {series.max():.4f})"
    )


for action_type in ["OP", "SP"]:
    print(f"\n=== {action_type} actions ===")
    print_metric_interpretation(action_type, "merged_ctx_03LIC_1071_pv_at_action", "1071 PV at action")
    print_metric_interpretation(action_type, "merged_ctx_alarm_proximity", "alarm proximity")
    print_metric_interpretation(action_type, "merged_ctx_time_progress_ratio", "time progress ratio")
    print_metric_interpretation(action_type, "merged_ctx_03LIC_1071_norm_pos", "1071 normalized position")
    print_metric_interpretation(action_type, "merged_ctx_03LIC_1071_episode_norm_roc", "episode normalized change")
    print_metric_interpretation(action_type, "merged_ctx_03LIC_1071_local_3m_delta_norm", "local 3 min normalized change")
    print_metric_interpretation(action_type, "merged_ctx_03LIC_1071_local_5m_delta_norm", "local 5 min normalized change")

print("\nInterpretation guide:")
print("- 1071 PV at action: absolute target PV level when the merged action starts.")
print("- Alarm proximity: 0 means exactly at the low alarm threshold, positive means above threshold, negative means below threshold.")
print("- Time progress ratio: action timing relative to the median cluster duration; <1 earlier, >1 later.")
print("- 1071 normalized position: PV location within the operating range; near 0 is near lower bound, near 1 is near upper bound.")
print("- Episode normalized change: change from deviation start to action time, scaled by the operating range.")
print("- Local 3/5 min normalized change: recent short-window movement before the action; positive means rising, negative means falling.")


=== OP actions ===
OP | 1071 PV at action: median=29.3790, IQR=(1.8519, 58.3740), range=(-1.3276, 103.1308)
OP | alarm proximity: median=0.0460, IQR=(-1.9684, 2.1679), range=(-2.2011, 5.4432)
OP | time progress ratio: median=3.0954, IQR=(2.4467, 4.2606), range=(0.3517, 8.7086)
OP | 1071 normalized position: median=-0.8191, IQR=(-4.6604, 3.2270), range=(-5.1041, 9.4726)
OP | episode normalized change: median=-1.1482, IQR=(-4.8507, 2.5203), range=(-13.8162, 11.5859)
OP | local 3 min normalized change: median=-0.0007, IQR=(-0.9313, 0.8465), range=(-14.0426, 14.4529)
OP | local 5 min normalized change: median=-0.0006, IQR=(-1.2768, 1.0477), range=(-9.1332, 14.5611)

=== SP actions ===
SP | 1071 PV at action: median=33.4924, IQR=(27.6213, 52.5443), range=(-1.2993, 75.0761)
SP | alarm proximity: median=0.3471, IQR=(-0.0826, 1.7413), range=(-2.1990, 3.3902)
SP | time progress ratio: median=3.4641, IQR=(2.5602, 4.3807), range=(1.5316, 12.3237)
SP | 1071 normalized position: median=-0.2451, IQ

## Plant-context feature builder (3 / 10 / 30 min windows)

Computes "dimensions" as a single per-merged-action feature
table (`pm_features_df`), using short / medium / long look-back windows of
**3 / 10 / 30 minutes**.

**Context features** (the similarity key — all computable at runtime from live PV
history + operating limits + a deviation-start time), produced for the alarm tag and
every related PV tag:
- current PV (raw) and normalized position in the operating band
- deviation-start PV (raw) and its normalized position
- change since deviation start (normalized by operating range)
- ROC over ST / MT / LT windows (normalized by operating range)
- ROC direction over ST / MT / LT (`increasing` / `decreasing` / `same`)
- trajectory vs the nearest operating limit (`towards_limit` / `away_from_limit` / `stable`)
- alarm proximity and minutes-since-deviation (alarm tag only)

**Action payload** (kept for the record but **not** similarity inputs — these are the
*outcome* we retrieve, unknown at decision time): action tag + parameter, signed step,
direction, number of step changes, and time between consecutive step changes.

All ROC/position features are range-normalized so different tags are comparable, and
nothing here uses look-ahead information. This block is standalone; it builds
`pm_features_df` for inspection and does not modify the enriched workbook above.


In [7]:
# ============================================================================
# PM DIMENSIONS — plant-context feature builder (3 / 10 / 30 min windows)
# ============================================================================
# Reuses objects built above: pv_op_data_df, valid_pv_tags, lower_bounds, ranges,
# operating_limits_df, group_meta_df, action_snapshot, deviation_snapshot,
# asof_snapshot, TARGET_SSD_TAG, ALARM_THRESHOLD.

ROC_WINDOWS = {"st": 3, "mt": 10, "lt": 30}   # look-back minutes (short / medium / long)
ROC_DEADBAND = 0.002                          # |normalized ROC| below this -> "same" / "stable"

lower_arr = lower_bounds.values
range_arr = ranges.values

# --- PV snapshot of every valid tag at (action_time - W) for each window -----
pm_window_snapshots = {
    name: asof_snapshot(
        pv_op_data_df[valid_pv_tags],
        group_meta_df["merged_action_timestamp"] - pd.Timedelta(minutes=minutes),
    )
    for name, minutes in ROC_WINDOWS.items()
}

# --- core per-tag frames (row order aligned 1:1 with group_meta_df) ----------
pm_cur_pv_df = action_snapshot[valid_pv_tags]
pm_dev_pv_df = deviation_snapshot[valid_pv_tags]

pm_norm_pos_df       = (pm_cur_pv_df - lower_arr) / range_arr
pm_dev_norm_pos_df   = (pm_dev_pv_df - lower_arr) / range_arr
pm_episode_change_df = (pm_cur_pv_df - pm_dev_pv_df) / range_arr

pm_roc_norm = {
    name: (pm_cur_pv_df - pm_window_snapshots[name][valid_pv_tags]) / range_arr
    for name in ROC_WINDOWS
}


def _roc_direction(norm_roc_df, deadband=ROC_DEADBAND):
    """Elementwise increasing / decreasing / same on a normalized-ROC frame."""
    labels = np.where(
        norm_roc_df.values > deadband, "increasing",
        np.where(norm_roc_df.values < -deadband, "decreasing", "same"),
    )
    return pd.DataFrame(labels, index=norm_roc_df.index, columns=norm_roc_df.columns)


def _trajectory_vs_limit(norm_pos_df, norm_roc_df, deadband=ROC_DEADBAND):
    """towards_limit / away_from_limit / stable relative to the NEAREST limit
    (nearest chosen by normalized position: <= 0.5 -> lower bound, else upper)."""
    nearest_lower = norm_pos_df.values <= 0.5
    roc = norm_roc_df.values
    towards = np.where(nearest_lower, roc < -deadband, roc > deadband)
    labels = np.where(
        np.abs(roc) <= deadband, "stable",
        np.where(towards, "towards_limit", "away_from_limit"),
    )
    return pd.DataFrame(labels, index=norm_pos_df.index, columns=norm_pos_df.columns)


pm_roc_dir = {name: _roc_direction(pm_roc_norm[name]) for name in ROC_WINDOWS}
pm_traj_df = _trajectory_vs_limit(pm_norm_pos_df, pm_roc_norm["st"])   # short-term movement

print(f"Merged actions (rows)       : {len(group_meta_df):,}")
print(f"Context tags (alarm+related): {len(valid_pv_tags)}  (target present: {TARGET_SSD_TAG in valid_pv_tags})")
print(f"Windows (min)               : ST={ROC_WINDOWS['st']}, MT={ROC_WINDOWS['mt']}, LT={ROC_WINDOWS['lt']}")
print(f"Actions with deviation_start: {group_meta_df['deviation_start'].notna().sum():,} / {len(group_meta_df):,}")


Merged actions (rows)       : 7,246
Context tags (alarm+related): 26  (target present: True)
Windows (min)               : ST=3, MT=10, LT=30
Actions with deviation_start: 6,634 / 7,246


In [8]:
# --- flatten per-tag frames into <tag>_<suffix> columns ---------------------
def _flatten(frame, suffix):
    out = frame.reset_index(drop=True).copy()
    out.columns = [f"{col.replace('.PV', '')}_{suffix}" for col in out.columns]
    return out


pm_context_df = pd.concat(
    [
        _flatten(pm_cur_pv_df, "pv_now"),
        _flatten(pm_norm_pos_df, "norm_pos"),
        _flatten(pm_dev_pv_df, "pv_dev_start"),
        _flatten(pm_dev_norm_pos_df, "dev_norm_pos"),
        _flatten(pm_episode_change_df, "episode_change"),
        _flatten(pm_roc_norm["st"], "roc_st"),
        _flatten(pm_roc_norm["mt"], "roc_mt"),
        _flatten(pm_roc_norm["lt"], "roc_lt"),
        _flatten(pm_roc_dir["st"], "roc_dir_st"),
        _flatten(pm_roc_dir["mt"], "roc_dir_mt"),
        _flatten(pm_roc_dir["lt"], "roc_dir_lt"),
        _flatten(pm_traj_df, "traj_vs_limit"),
    ],
    axis=1,
)

# alarm-tag-only extras
target_stub = TARGET_SSD_TAG.replace(".PV", "")
if TARGET_SSD_TAG in pm_cur_pv_df.columns:
    target_upper = operating_limits_df.loc[TARGET_SSD_TAG, "UPPER_LIMIT"]
    pm_context_df[f"{target_stub}_alarm_proximity"] = (
        pm_cur_pv_df[TARGET_SSD_TAG].to_numpy() - ALARM_THRESHOLD
    ) / (target_upper - ALARM_THRESHOLD)
pm_context_df["minutes_since_deviation"] = group_meta_df["merged_minutes_from_deviation"].to_numpy()

# --- action payload (retrieved, NOT similarity inputs) ----------------------
# time between consecutive step changes + step count, per tag+parameter within a cluster
_seq = group_meta_df[
    ["merged_group_id", "cluster_id", "Source", "Description", "merged_action_timestamp"]
].copy()
_seq = _seq.sort_values(["cluster_id", "Source", "Description", "merged_action_timestamp"])
_seq["minutes_since_prev_step"] = (
    _seq.groupby(["cluster_id", "Source", "Description"])["merged_action_timestamp"]
    .diff().dt.total_seconds() / 60.0
)
_seq["cluster_tag_num_steps"] = (
    _seq.groupby(["cluster_id", "Source", "Description"])["merged_group_id"].transform("size")
)
_pace = _seq.set_index("merged_group_id")[["minutes_since_prev_step", "cluster_tag_num_steps"]]

action_payload_cols = [
    "merged_group_id", "cluster_id", "Source", "Description",
    "merged_action_timestamp", "merged_prev_value", "merged_value",
    "merged_step", "merged_action_direction", "merged_num_actions",
    "deviation_start", "merged_minutes_from_deviation",
]

# --- assemble the single per-merged-action feature table --------------------
pm_features_df = pd.concat(
    [group_meta_df[action_payload_cols].reset_index(drop=True),
     pm_context_df.reset_index(drop=True)],
    axis=1,
)
pm_features_df["minutes_since_prev_step"] = pm_features_df["merged_group_id"].map(
    _pace["minutes_since_prev_step"]
)
pm_features_df["cluster_tag_num_steps"] = pm_features_df["merged_group_id"].map(
    _pace["cluster_tag_num_steps"]
)

context_feature_cols = list(pm_context_df.columns)
payload_feature_cols = [c for c in action_payload_cols if c != "merged_group_id"] + [
    "minutes_since_prev_step", "cluster_tag_num_steps"
]

print(f"pm_features_df shape         : {pm_features_df.shape[0]:,} rows x {pm_features_df.shape[1]:,} cols")
print(f"Context (similarity) columns : {len(context_feature_cols)}")
print(f"Action payload columns       : {len(payload_feature_cols)}")

# --- quick sanity look at the alarm tag -------------------------------------
target_preview_cols = [
    f"{target_stub}_pv_now", f"{target_stub}_norm_pos", f"{target_stub}_pv_dev_start",
    f"{target_stub}_episode_change", f"{target_stub}_roc_st", f"{target_stub}_roc_mt",
    f"{target_stub}_roc_lt", f"{target_stub}_roc_dir_st", f"{target_stub}_traj_vs_limit",
    f"{target_stub}_alarm_proximity", "minutes_since_deviation",
]
print("\nAlarm-tag ST ROC direction distribution:")
print(pm_features_df[f"{target_stub}_roc_dir_st"].value_counts(dropna=False).to_string())
print("\nAlarm-tag trajectory-vs-limit distribution:")
print(pm_features_df[f"{target_stub}_traj_vs_limit"].value_counts(dropna=False).to_string())
display(pm_features_df[["merged_group_id", "Source", "Description"] + target_preview_cols].head(10))


pm_features_df shape         : 7,246 rows x 328 cols
Context (similarity) columns : 314
Action payload columns       : 13

Alarm-tag ST ROC direction distribution:
03LIC_1071_roc_dir_st
decreasing    3729
increasing    3175
same           342

Alarm-tag trajectory-vs-limit distribution:
03LIC_1071_traj_vs_limit
towards_limit      4427
away_from_limit    2479
stable              340


,merged_group_id,Source,Description,03LIC_1071_pv_now,03LIC_1071_norm_pos,03LIC_1071_pv_dev_start,03LIC_1071_episode_change,03LIC_1071_roc_st,03LIC_1071_roc_mt,03LIC_1071_roc_lt,03LIC_1071_roc_dir_st,03LIC_1071_traj_vs_limit,03LIC_1071_alarm_proximity,minutes_since_deviation
0,MG_00001,03FIC_1085,OP,18.835490,-2.290387,34.68777,-2.212109,-0.754668,-4.411418,-2.377010,decreasing,towards_limit,-0.725548,89.744267
1,MG_00002,03FIC_1085,OP,24.066044,-1.560488,34.68777,-1.482211,1.389627,3.266101,0.943048,increasing,away_from_limit,-0.342774,120.913450
2,MG_00003,03FIC_1085,OP,26.175201,-1.266165,34.68777,-1.187888,1.214412,3.597363,1.578079,increasing,away_from_limit,-0.188425,121.003633
3,MG_00004,03FIC_1085,OP,28.495543,-0.942373,34.68777,-0.864095,1.061617,3.946646,2.377102,increasing,away_from_limit,-0.018621,122.024483
4,MG_00005,03FIC_1085,OP,32.242134,-0.419554,34.68777,-0.341277,0.522819,3.620499,3.782968,increasing,away_from_limit,0.255556,125.134267
5,MG_00006,03FIC_3435,OP,47.546093,1.716040,34.68777,1.794317,1.996345,1.542215,1.812659,increasing,towards_limit,1.375506,72.871717
6,MG_00007,03FIC_3435,OP,62.256752,3.768841,34.68777,3.847119,3.091482,3.735517,3.831970,increasing,towards_limit,2.452039,74.625883
7,MG_00008,03FIC_3435,OP,33.197815,-0.286193,34.68777,-0.207916,0.278548,0.424623,4.540396,increasing,away_from_limit,0.325493,140.551767
8,MG_00009,03HIC_1141,OP,40.102764,0.677359,34.68777,0.755636,0.766041,0.518894,0.790478,increasing,towards_limit,0.830800,71.997850
9,MG_00010,03HIC_1141,OP,47.546093,1.716040,34.68777,1.794317,1.996345,1.542215,1.812659,increasing,towards_limit,1.375506,72.154300
